In [25]:
import os
import numpy as np
import cupy as cp
from scipy.signal import butter, sosfilt_zi, sosfilt as cpu_sosfilt
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

###Configuration

In [ ]:
INPUT_DIR  = r"C:\Users\RF-LAB\Desktop\Pipeline\valid_test_iq"
OUTPUT_DIR = r"C:\Users\RF-LAB\Desktop\Pipeline\valid_test_processed"

fs           = 100e6
lowcut       = 100e3
highcut      = 45e6
order        = 5
BINARY_DTYPE = np.float32

SUPPORTED_EXTENSIONS = {".iq", ".bin", ".dat", ".csv"}

CHUNK_SIZE  = 5_000_000
MAX_WORKERS = 2


In [27]:
# ============================================================
# Filter builder
# ============================================================

def build_filter(fs, lowcut, highcut, order):
    nyq  = 0.5 * fs
    low  = max(lowcut  / nyq, 1e-6)
    high = min(highcut / nyq, 1.0 - 1e-6)
    sos  = butter(order, [low, high], btype='band', output='sos')
    return sos.astype(np.float32)

In [28]:
# ============================================================
# GPU check
# ============================================================

def check_gpu():
    try:
        props = cp.cuda.runtime.getDeviceProperties(0)
        name  = props['name'].decode()
        vram  = props['totalGlobalMem'] / 1e9
        print(f"  GPU detected : {name}")
        print(f"  VRAM         : {vram:.1f} GB")
        print(f"  Note         : GPU used for DC removal + RMS norm")
        print(f"                 Filtering on CPU (CuPy 13.x sosfilt bug)")
        return True
    except Exception as e:
        print(f"  [WARNING] CUDA not available: {e}")
        return False

In [29]:
def load_and_process(filepath, sos_cpu, chunk_size=CHUNK_SIZE):
    """
    Pipeline per chunk:
      1. CPU  — read raw bytes from SSD
      2. CPU  — de-interleave I and Q
      3. GPU  — DC removal (mean subtraction via CuPy)
      4. CPU  — bandpass filter via scipy sosfilt with state carry
      5. Accumulate into output buffers
    After all chunks:
      6. GPU  — RMS normalization
      7. CPU  — save as .npy
    """

    file_size     = os.path.getsize(filepath)
    bytes_per_val = np.dtype(BINARY_DTYPE).itemsize
    total_vals    = file_size // bytes_per_val
    if total_vals % 2 != 0:
        total_vals -= 1
    n_iq_pairs = total_vals // 2

    if n_iq_pairs == 0:
        raise ValueError(f"File empty or dtype mismatch: {filepath}")

    print(f"    Samples : {n_iq_pairs:,} IQ pairs  "
          f"({n_iq_pairs/fs:.2f}s of signal)")

    I_out = np.empty(n_iq_pairs, dtype=np.float32)
    Q_out = np.empty(n_iq_pairs, dtype=np.float32)

    # Filter state — carried across chunks
    zi_I = sosfilt_zi(sos_cpu).astype(np.float32)
    zi_Q = sosfilt_zi(sos_cpu).astype(np.float32)

    write_pos = 0

    with open(filepath, 'rb') as f:
        while True:
            raw_bytes = f.read(chunk_size * 2 * bytes_per_val)
            if not raw_bytes:
                break

            raw = np.frombuffer(raw_bytes, dtype=BINARY_DTYPE).copy()
            if len(raw) % 2 != 0:
                raw = raw[:-1]
            if len(raw) == 0:
                break

            n_pairs = len(raw) // 2

            # De-interleave on CPU
            I_chunk = raw[0::2].astype(np.float32)
            Q_chunk = raw[1::2].astype(np.float32)

            # DC removal on GPU — fast mean subtraction
            I_gpu    = cp.asarray(I_chunk)
            Q_gpu    = cp.asarray(Q_chunk)
            I_gpu   -= cp.mean(I_gpu)
            Q_gpu   -= cp.mean(Q_gpu)
            I_chunk  = cp.asnumpy(I_gpu)
            Q_chunk  = cp.asnumpy(Q_gpu)
            del I_gpu, Q_gpu
            cp.get_default_memory_pool().free_all_blocks()

            # Bandpass filter on CPU with state carry-over
            I_filt, zi_I = cpu_sosfilt(sos_cpu, I_chunk, zi=zi_I)
            Q_filt, zi_Q = cpu_sosfilt(sos_cpu, Q_chunk, zi=zi_Q)

            I_out[write_pos:write_pos + n_pairs] = I_filt
            Q_out[write_pos:write_pos + n_pairs] = Q_filt
            write_pos += n_pairs

    I_out = I_out[:write_pos]
    Q_out = Q_out[:write_pos]

    # RMS normalization on GPU
    I_gpu = cp.asarray(I_out)
    Q_gpu = cp.asarray(Q_out)
    rms   = cp.sqrt(cp.mean(I_gpu**2 + Q_gpu**2))

    if float(rms) > 1e-10:
        I_gpu /= rms
        Q_gpu /= rms
    else:
        print("    [WARNING] Near-zero RMS — file may be silent.")

    I_out = cp.asnumpy(I_gpu)
    Q_out = cp.asnumpy(Q_gpu)
    del I_gpu, Q_gpu
    cp.get_default_memory_pool().free_all_blocks()

    return I_out, Q_out

In [30]:
# ============================================================
# Per-file worker
# ============================================================

def process_single_file(args):
    fname, input_path, output_path, fs, lowcut, highcut, order = args
    t_start = time.time()

    try:
        sos  = build_filter(fs, lowcut, highcut, order)
        I, Q = load_and_process(input_path, sos)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        np.save(output_path, np.stack([I, Q], axis=0))

        n_samples = len(I)
        size_mb   = os.path.getsize(input_path) / 1e6
        duration  = time.time() - t_start
        speed     = size_mb / duration

        return (fname, True,
                f"{n_samples:,} samples | "
                f"{size_mb:.0f} MB | "
                f"{duration:.1f}s | "
                f"{speed:.1f} MB/s",
                duration)

    except Exception as e:
        import traceback
        return (fname, False,
                f"{e}\n{traceback.format_exc()}",
                time.time() - t_start)


In [31]:
# ============================================================
# Core: chunked load + filter + GPU normalize
# ============================================================

def load_and_process(filepath, sos_cpu, chunk_size=CHUNK_SIZE):

    # ── Detect file format ────────────────────────────────
    with open(filepath, 'rb') as f:
        header = f.read(32)
    try:
        header.decode('utf-8')
        is_text = True
    except UnicodeDecodeError:
        is_text = False

    # ── Load I, Q based on format ─────────────────────────
    if is_text:
        # DroneRF text CSV — read all at once (fast with fromfile)
        data = np.fromfile(filepath, dtype=np.float32, sep=',')
        data = data.reshape(-1, 2)
        I_out_raw = data[:, 0]
        Q_out_raw = data[:, 1]
        n_iq_pairs = len(I_out_raw)
        print(f"    Samples : {n_iq_pairs:,} IQ pairs  "
              f"({n_iq_pairs/fs:.2f}s of signal)  [DroneRF CSV]")

        # DC removal on GPU
        I_gpu = cp.asarray(I_out_raw)
        Q_gpu = cp.asarray(Q_out_raw)
        I_gpu -= cp.mean(I_gpu)
        Q_gpu -= cp.mean(Q_gpu)

        # Bandpass filter on CPU
        I_filt, _ = cpu_sosfilt(sos_cpu, cp.asnumpy(I_gpu),
                                 zi=sosfilt_zi(sos_cpu).astype(np.float32))
        Q_filt, _ = cpu_sosfilt(sos_cpu, cp.asnumpy(Q_gpu),
                                 zi=sosfilt_zi(sos_cpu).astype(np.float32))
        del I_gpu, Q_gpu
        cp.get_default_memory_pool().free_all_blocks()

        I_out = I_filt
        Q_out = Q_filt

    else:
        # ── Your original binary chunked pipeline ─────────
        file_size     = os.path.getsize(filepath)
        bytes_per_val = np.dtype(BINARY_DTYPE).itemsize
        total_vals    = file_size // bytes_per_val
        if total_vals % 2 != 0:
            total_vals -= 1
        n_iq_pairs = total_vals // 2
        if n_iq_pairs == 0:
            raise ValueError(f"File empty or dtype mismatch: {filepath}")
        print(f"    Samples : {n_iq_pairs:,} IQ pairs  "
              f"({n_iq_pairs/fs:.2f}s of signal)")

        I_out = np.empty(n_iq_pairs, dtype=np.float32)
        Q_out = np.empty(n_iq_pairs, dtype=np.float32)
        zi_I = sosfilt_zi(sos_cpu).astype(np.float32)
        zi_Q = sosfilt_zi(sos_cpu).astype(np.float32)
        write_pos = 0

        with open(filepath, 'rb') as f:
            while True:
                raw_bytes = f.read(chunk_size * 2 * bytes_per_val)
                if not raw_bytes:
                    break
                raw = np.frombuffer(raw_bytes, dtype=BINARY_DTYPE).copy()
                if len(raw) % 2 != 0:
                    raw = raw[:-1]
                if len(raw) == 0:
                    break
                n_pairs = len(raw) // 2
                I_chunk = raw[0::2].astype(np.float32)
                Q_chunk = raw[1::2].astype(np.float32)
                I_gpu   = cp.asarray(I_chunk)
                Q_gpu   = cp.asarray(Q_chunk)
                I_gpu  -= cp.mean(I_gpu)
                Q_gpu  -= cp.mean(Q_gpu)
                I_chunk = cp.asnumpy(I_gpu)
                Q_chunk = cp.asnumpy(Q_gpu)
                del I_gpu, Q_gpu
                cp.get_default_memory_pool().free_all_blocks()
                I_filt, zi_I = cpu_sosfilt(sos_cpu, I_chunk, zi=zi_I)
                Q_filt, zi_Q = cpu_sosfilt(sos_cpu, Q_chunk, zi=zi_Q)
                I_out[write_pos:write_pos + n_pairs] = I_filt
                Q_out[write_pos:write_pos + n_pairs] = Q_filt
                write_pos += n_pairs

        I_out = I_out[:write_pos]
        Q_out = Q_out[:write_pos]

    # ── RMS normalization on GPU (same for both formats) ──
    I_gpu = cp.asarray(I_out)
    Q_gpu = cp.asarray(Q_out)
    rms   = cp.sqrt(cp.mean(I_gpu**2 + Q_gpu**2))
    if float(rms) > 1e-10:
        I_gpu /= rms
        Q_gpu /= rms
    else:
        print("    [WARNING] Near-zero RMS — file may be silent.")
    I_out = cp.asnumpy(I_gpu)
    Q_out = cp.asnumpy(Q_gpu)
    del I_gpu, Q_gpu
    cp.get_default_memory_pool().free_all_blocks()

    return I_out, Q_out

In [32]:
def process_single_file(args):
    fname, input_path, output_path, fs, lowcut, highcut, order = args
    t_start = time.time()

    try:
        sos  = build_filter(fs, lowcut, highcut, order)
        I, Q = load_and_process(input_path, sos)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        np.save(output_path, np.stack([I, Q], axis=0))

        n_samples = len(I)
        size_mb   = os.path.getsize(input_path) / 1e6
        duration  = time.time() - t_start
        speed     = size_mb / duration

        return (fname, True,
                f"{n_samples:,} samples | "
                f"{size_mb:.0f} MB | "
                f"{duration:.1f}s | "
                f"{speed:.1f} MB/s",
                duration)

    except Exception as e:
        import traceback
        return (fname, False,
                f"{e}\n{traceback.format_exc()}",
                time.time() - t_start)

In [1]:
import numpy as np

# Check one file
test_file = r"C:\Users\RF-LAB\Desktop\Pipeline\valid_test_iq\1692845586.500756.npy"
data = np.load(test_file)
print(f"Shape : {data.shape}")
print(f"Dtype : {data.dtype}")
print(f"Is complex : {np.iscomplexobj(data)}")
print(f"Sample values : {data[:5]}")
print(f"Min: {data.min():.4f}  Max: {data.max():.4f}  Mean: {data.mean():.4f}")

Shape : (258, 3)
Dtype : float64
Is complex : False
Sample values : [[-4.33841753  2.18773651  4.23285675]
 [-4.3827405  -1.59462202  4.60104418]
 [ 7.68118525  1.16943562  7.10011625]
 [ 8.27112579 -1.52611864  7.18552065]
 [ 8.35142803 -1.5409354   7.25528288]]
Min: -4.4223  Max: 8.4209  Mean: 1.4622


In [2]:
import numpy as np
import os

# Check a few files to see if shape is consistent
folder = r"C:\Users\RF-LAB\Desktop\Pipeline\valid_test_iq"
files = [f for f in os.listdir(folder) if f.endswith('.npy')][:5]

for f in files:
    data = np.load(os.path.join(folder, f))
    print(f"{f} → shape: {data.shape}, dtype: {data.dtype}, mean: {data.mean():.3f}")

1692845586.500756.npy → shape: (258, 3), dtype: float64, mean: 1.462
1692845586.566210.npy → shape: (258, 3), dtype: float64, mean: 1.462
1692845586.634495.npy → shape: (256, 3), dtype: float64, mean: 1.438
1692845586.699294.npy → shape: (256, 3), dtype: float64, mean: 1.438
1692845586.767452.npy → shape: (251, 3), dtype: float64, mean: 1.416


In [33]:
def run_preprocessing():

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print("=" * 65)
    print("  IQ PREPROCESSING PIPELINE")
    print("=" * 65)
    use_gpu = check_gpu()

    all_files = []
    for root, dirs, files in os.walk(INPUT_DIR):
        for f in files:
            if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS:
                all_files.append(os.path.join(root, f))

    if not all_files:
        print(f"  No supported files found in: {INPUT_DIR}")
        return

    task_args = []
    for fpath in all_files:
        fname       = os.path.basename(fpath)
        rel_path    = os.path.relpath(fpath, INPUT_DIR)
        base_name   = os.path.splitext(rel_path)[0]
        output_path = os.path.join(OUTPUT_DIR,
                                    base_name + "_processed.npy")
        task_args.append((
            fname, fpath, output_path,
            fs, lowcut, highcut, order
        ))

    total_gb = sum(os.path.getsize(a[1]) for a in task_args) / 1e9

    print(f"  Input dir   : {INPUT_DIR}")
    print(f"  Output dir  : {OUTPUT_DIR}")
    print(f"  Sample rate : {fs/1e6:.0f} MHz")
    print(f"  Bandpass    : {lowcut/1e3:.0f} kHz → {highcut/1e6:.0f} MHz")
    print(f"  Chunk size  : {CHUNK_SIZE/1e6:.0f}M samples")
    print(f"  Workers     : {MAX_WORKERS} threads")
    print(f"  Files       : {len(all_files)}  |  Total: {total_gb:.1f} GB")
    print("=" * 65)
    print()

    success  = 0
    skipped  = 0
    t_total  = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_single_file, args): args[0]
            for args in task_args
        }
        for future in as_completed(futures):
            fname, ok, msg, duration = future.result()
            status = "✓ OK  " if ok else "✗ FAIL"
            print(f"  [{status}] {fname}")
            print(f"           {msg}")
            print()
            if ok:
                success += 1
            else:
                skipped += 1

    total_time = time.time() - t_total
    print("=" * 65)
    print(f"  COMPLETE")
    print(f"  Success : {success}  |  Failed : {skipped}")
    print(f"  Total   : {total_time/60:.1f} min")
    print(f"  Output  : {OUTPUT_DIR}")
    print("=" * 65)


if __name__ == "__main__":
    run_preprocessing()

  IQ PREPROCESSING PIPELINE
  GPU detected : Tesla T4
  VRAM         : 16.1 GB
  Note         : GPU used for DC removal + RMS norm
                 Filtering on CPU (CuPy 13.x sosfilt bug)
  No supported files found in: C:\Users\RF-LAB\Desktop\Pipeline\valid_test_iq


In [34]:
def load_dronerf_csv(filepath):
    data = np.fromfile(filepath, dtype=np.float32, sep=',')
    # Reshape into (N, 2) — alternating I, Q
    data = data.reshape(-1, 2)
    I = data[:, 0]
    Q = data[:, 1]
    return I, Q

In [ ]:
# Test on one failing file — change path to match yours
test_file = r"C:\Users\RF-LAB\Desktop\Pipeline\Train_iq\Bepop drone\RF Data_10000_L\RF Data_10000_L\10000L_5.csv"

I, Q = load_dronerf_csv(test_file)
print(f"I shape : {I.shape}")
print(f"Q shape : {Q.shape}")
print(f"I sample values : {I[:5]}")
print(f"Q sample values : {Q[:5]}")

In [ ]:
import os

OUTPUT_DIR = r"C:\Users\RF-LAB\Desktop\Pipeline\Train_iq_preprocessed"

npy_files = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        if f.endswith('_processed.npy'):
            npy_files.append(f)

print(f"Total processed .npy files : {len(npy_files)}")

nodrone = [f for f in npy_files if f.startswith('00000')]
drone   = [f for f in npy_files if not f.startswith('00000')]

print(f"Drone files    : {len(drone)}")
print(f"No-drone files : {len(nodrone)}")

Total processed .npy files : 454
Drone files    : 372
No-drone files : 82
